# Run Duration Valuation Phases

Execute only notebooks `04` through `07` for the selected BESS durations: `1h`, `2h`, and `4h`. Executed copies are written to `data/processed/notebook_runs/<timestamp>/` so source notebooks are not overwritten.

This runner assumes Phases `01`-`03` have already produced the shared raw, calibration, and simulation artefacts.


In [1]:
import json
import os
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

# Bootstrap imports from the project root, then use the shared helper.
_ROOT_CANDIDATE = Path.cwd()
for candidate in [_ROOT_CANDIDATE, *_ROOT_CANDIDATE.parents]:
    if (candidate / "src").is_dir() and (candidate / "data").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError("Could not find project root containing src/ and data/.")

from src.utils import find_project_root

PROJECT_ROOT = find_project_root(_ROOT_CANDIDATE)
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_runs" / datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_NOTEBOOKS = [
    "04_lsmc_valuation.ipynb",
    "05_mtm_risk_greeks.ipynb",
    "06_backtest_pnl.ipynb",
    "07_historical_perfect_foresight.ipynb",
]
VALUATION_DURATIONS_H = [1.0, 2.0, 4.0]

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Executed notebooks dir: {OUTPUT_DIR}")


Project root: G:\My Drive\Research\bess_project
Notebook dir: G:\My Drive\Research\bess_project\notebooks
Executed notebooks dir: G:\My Drive\Research\bess_project\data\processed\notebook_runs\20260501_001223


In [2]:
missing = [name for name in PHASE_NOTEBOOKS if not (NOTEBOOK_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing phase notebooks: {missing}")

print("Run matrix:")
for duration_h in VALUATION_DURATIONS_H:
    label = f"{duration_h:g}h"
    for name in PHASE_NOTEBOOKS:
        print(f"- {label}: {name}")


Run matrix:
- 1h: 04_lsmc_valuation.ipynb
- 1h: 05_mtm_risk_greeks.ipynb
- 1h: 06_backtest_pnl.ipynb
- 1h: 07_historical_perfect_foresight.ipynb
- 2h: 04_lsmc_valuation.ipynb
- 2h: 05_mtm_risk_greeks.ipynb
- 2h: 06_backtest_pnl.ipynb
- 2h: 07_historical_perfect_foresight.ipynb
- 4h: 04_lsmc_valuation.ipynb
- 4h: 05_mtm_risk_greeks.ipynb
- 4h: 06_backtest_pnl.ipynb
- 4h: 07_historical_perfect_foresight.ipynb


In [3]:
def parameterized_notebook(src_path: Path, duration_h: float, work_dir: Path) -> Path:
    """Create a temporary notebook with VALUATION_DURATION_H injected as the first code cell."""
    nb = json.loads(src_path.read_text(encoding="utf-8"))
    label = f"{duration_h:g}h"
    param_cell = {
        "cell_type": "code",
        "execution_count": None,
        "metadata": {"tags": ["parameters"]},
        "outputs": [],
        "source": [
            f"VALUATION_DURATION_H = {duration_h!r}\n",
            "PHASE4_RUN_MODE = 'partial'\n",
            f"print('Injected valuation duration: {label}')\n",
            "print('Injected Phase 4 run mode: medium')\n",
        ],
    }
    nb["cells"] = [param_cell, *nb.get("cells", [])]
    tmp_path = work_dir / f"{src_path.stem}_{label}.parameterized.ipynb"
    tmp_path.write_text(json.dumps(nb, indent=1), encoding="utf-8")
    return tmp_path


results = []
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT) + os.pathsep + env.get("PYTHONPATH", "")

for duration_h in VALUATION_DURATIONS_H:
    label = f"{duration_h:g}h"
    print("\n" + "#" * 80)
    print(f"Running phases 04-07 for {label} BESS")
    print("#" * 80)

    for phase_name in PHASE_NOTEBOOKS:
        src = NOTEBOOK_DIR / phase_name
        tmp_src = parameterized_notebook(src, duration_h, OUTPUT_DIR)
        out_name = f"{src.stem}_{label}.executed.ipynb"
        cmd = [
            sys.executable,
            "-m",
            "jupyter",
            "nbconvert",
            "--to",
            "notebook",
            "--execute",
            "--ExecutePreprocessor.timeout=-1",
            "--ExecutePreprocessor.kernel_name=python3",
            f"--output-dir={OUTPUT_DIR}",
            f"--output={out_name}",
            str(tmp_src),
        ]

        print("\n" + "=" * 80)
        print(f"Executing {phase_name} for {label}")
        print("=" * 80)
        started = time.time()
        proc = subprocess.run(
            cmd,
            cwd=PROJECT_ROOT,
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        elapsed = time.time() - started
        results.append({
            "duration_h": duration_h,
            "duration_label": label,
            "notebook": phase_name,
            "returncode": proc.returncode,
            "elapsed_s": elapsed,
            "output": out_name,
        })
        print(proc.stdout[-4000:])
        print(f"Finished {phase_name} for {label} in {elapsed:.1f}s with return code {proc.returncode}")
        if proc.returncode != 0:
            raise RuntimeError(f"Execution failed: {phase_name} for {label}")

print("\nAll requested duration phase notebooks completed.")



################################################################################
Running phases 04-07 for 1h BESS
################################################################################

Executing 04_lsmc_valuation.ipynb for 1h
[NbConvertApp] Converting notebook G:\My Drive\Research\bess_project\data\processed\notebook_runs\20260501_001223\04_lsmc_valuation_1h.parameterized.ipynb to notebook
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\zmq\_future.py:718: RuntimeWarning: Proactor event loop does not implement add_reader fami

In [4]:
import pandas as pd

summary = pd.DataFrame(results)
summary["elapsed_min"] = summary["elapsed_s"] / 60.0
summary_path = OUTPUT_DIR / "run_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved run summary: {summary_path}")
summary


Saved run summary: G:\My Drive\Research\bess_project\data\processed\notebook_runs\20260501_001223\run_summary.csv


,duration_h,duration_label,notebook,returncode,elapsed_s,output,elapsed_min
0,1.0,1h,04_lsmc_valuation.ipynb,0,44.641594,04_lsmc_valuation_1h.executed.ipynb,0.744027
1,1.0,1h,05_mtm_risk_greeks.ipynb,0,13.681649,05_mtm_risk_greeks_1h.executed.ipynb,0.228027
2,1.0,1h,06_backtest_pnl.ipynb,0,56.663023,06_backtest_pnl_1h.executed.ipynb,0.944384
3,1.0,1h,07_historical_perfect_foresight.ipynb,0,22.421686,07_historical_perfect_foresight_1h.executed.ipynb,0.373695
4,2.0,2h,04_lsmc_valuation.ipynb,0,65.796054,04_lsmc_valuation_2h.executed.ipynb,1.096601
5,2.0,2h,05_mtm_risk_greeks.ipynb,0,11.149180,05_mtm_risk_greeks_2h.executed.ipynb,0.185820
6,2.0,2h,06_backtest_pnl.ipynb,0,81.334422,06_backtest_pnl_2h.executed.ipynb,1.355574
7,2.0,2h,07_historical_perfect_foresight.ipynb,0,20.965781,07_historical_perfect_foresight_2h.executed.ipynb,0.349430
8,4.0,4h,04_lsmc_valuation.ipynb,0,86.831738,04_lsmc_valuation_4h.executed.ipynb,1.447196
9,4.0,4h,05_mtm_risk_greeks.ipynb,0,13.653505,05_mtm_risk_greeks_4h.executed.ipynb,0.227558
